# Petrol Stations in Switzerland – Competitive Analysis

This notebook retrieves petrol station data for Switzerland from the OpenStreetMap Overpass API, saves the results as JSON and CSV, counts the number of stations per operator using a pivot table to identify the five largest operators, and visualises all stations on an interactive map.

## Libraries and settings

In [1]:
# Libraries
import os
import json
import requests
import folium
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

/workspaces/spatial_data_analysis/01_Python_Basic_GIS_Functionalities


## Fetch petrol station data from OpenStreetMap (Overpass API)

Note: If a JSONDecodeError occurs, try again this is most probably because the Overpass server is overloaded

In [5]:
overpass_urls = [                               #Liste mit Overpass-API-Endpunkten als Hauptserver und Fallback
    'https://overpass-api.de/api/interpreter',  #Hauptserver der Overpass API
    'https://overpass.osm.ch/api/interpreter',  #Schweizer Overpass-Server als Alternative
]

query = """                                     # Overpass-Abfrage für Tankstellen im Kanton Zürich definieren
[out:json];                                     #  Ausgabeformat JSON festlegen
area["ISO3166-2"="CH-ZH"][admin_level=4];       # Gebiet Kanton Zürich auswählen
node["amenity"="fuel"](area);                   # Alle OSM-Knoten mit amenity=fuel innerhalb des Gebiets suchen
out;                                            # Gefundene Elemente ausgeben
"""

import time                                                                             #Modul für Wartezeiten zwischen erneuten Anfragen importieren

max_retries = 5                                                                         #Maximale Anzahl Wiederholungsversuche pro Server festlegen
data = None                                                                             #Variable für die später empfangenen Daten initialisieren

for overpass_url in overpass_urls:                                                      #Overpass-Server nacheinander durchlaufen
    print(f'Trying Overpass endpoint: {overpass_url}')                                  #Aktuell verwendeten Server ausgeben

    for attempt in range(1, max_retries + 1):                                           #Mehrere Anfrageversuche pro Server durchführen
        try:                                                                            #Anfrage ausführen und mögliche Fehler abfangen
            response = requests.get(overpass_url, params={'data': query}, timeout=60)   #Anfrage mit Query an Overpass API senden
            data = response.json()                                                      #JSON-Antwort in Python-Datenstruktur umwandeln
            print(f"Number of petrol stations retrieved: {len(data['elements'])}")      #Anzahl gefundener Tankstellen ausgeben
            break                                                                       #Wiederholungsschleife bei erfolgreicher Anfrage verlassen

        except Exception:                                                               #Fehler bei Anfrage oder Verarbeitung abfangen
            if attempt < max_retries:                                                   #Prüfen, ob noch weitere Versuche möglich sind
                print("Server may be overloaded — retrying in 10 seconds...")           #Hinweis auf erneuten Versuch ausgeben
                time.sleep(10)                                                          #10 Sekunden vor dem nächsten Versuch warten
            else:                                                                       #Ausführen, wenn alle Versuche für diesen Server fehlgeschlagen sind
                print(f'Giving up on {overpass_url} after {max_retries} attempts.')     #Abbruch für aktuellen Server ausgeben

    if data is not None:                                                                #Prüfen, ob erfolgreich Daten empfangen wurden
        break                                                                           #Äussere Schleife beenden, wenn Daten vorhanden sind

if data is None:                                                                          #Prüfen, ob kein Server erfolgreich geantwortet hat
    raise RuntimeError("Max retries reached on all Overpass endpoints. Please try again later.")  #Fehlermeldung auslösen

Trying Overpass endpoint: https://overpass-api.de/api/interpreter
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Giving up on https://overpass-api.de/api/interpreter after 5 attempts.
Trying Overpass endpoint: https://overpass.osm.ch/api/interpreter
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Giving up on https://overpass.osm.ch/api/interpreter after 5 attempts.


RuntimeError: Max retries reached on all Overpass endpoints. Please try again later.

## Save raw data as JSON

In [ ]:
json_path = 'petrol_stations_ch.json'                   #Dateiname für die JSON-Datei festlegen

with open(json_path, 'w', encoding='utf-8') as f:       #JSON-Datei zum Schreiben öffnen
    json.dump(data, f, ensure_ascii=False, indent=2)    #Daten formatiert als JSON speichern

print(f'JSON saved to: {json_path}')                    #Speicherpfad der JSON-Datei ausgeben

## Parse JSON and create a DataFrame

In [6]:
records = []                                            #Leere Liste für die extrahierten Tankstellen-Datensätze erstellen

for element in data['elements']:                        #Alle gefundenen OSM-Elemente durchlaufen
    tags = element.get('tags', {})                      #Tag-Daten des Elements auslesen, falls vorhanden

    records.append({                                    #Relevante Informationen als Dictionary zur Liste hinzufügen
        'id': element.get('id'),                        #OSM-ID der Tankstelle übernehmen
        'lat': element.get('lat'),                      #Breitengrad übernehmen
        'lon': element.get('lon'),                      #Längengrad übernehmen
        'name': tags.get('name', 'unknown'),            #Name der Tankstelle auslesen
        'operator': tags.get('operator', 'unknown'),    #Betreiber der Tankstelle auslesen
        'brand': tags.get('brand', 'unknown'),          #Marke der Tankstelle auslesen
    })

df = pd.DataFrame(records)                              #Liste der Datensätze in einen Pandas DataFrame umwandeln

print(f'Shape: {df.shape}')                             #Anzahl Zeilen und Spalten des DataFrames ausgeben

df.head()                                               #Erste 5 Datensätze anzeigen

TypeError: 'NoneType' object is not subscriptable

## Save data as CSV

In [ ]:
csv_path = 'petrol_stations_ch.csv'                 #Dateiname für die CSV-Datei festlegen
df.to_csv(csv_path, index=False, encoding='utf-8')  #DataFrame ohne Index als UTF-8-CSV speichern
print(f'CSV saved to: {csv_path}')                  #Speicherpfad der CSV-Datei ausgeben

## Count petrol stations per operator (pivot table)

In [ ]:
pivot = df.pivot_table(                                                     #Pivot-Tabelle zur Anzahl Tankstellen pro Betreiber erstellen
    index='operator',                                                       #Betreiber als Zeilenindex verwenden
    values='id',                                                            #OSM-ID als zu zählende Spalte verwenden
    aggfunc='count'                                                         #Anzahl Einträge pro Betreiber zählen
).rename(columns={'id': 'count'}).sort_values('count', ascending=False)     #Spalte umbenennen und absteigend sortieren

print(f'Total operators: {len(pivot)}')                                     #Gesamtzahl unterschiedlicher Betreiber ausgeben

print(pivot.head(10))                                                       #Die 10 Betreiber mit den meisten Tankstellen anzeigen

## Top 5 largest operators

In [ ]:
top5 = pivot.head(5).reset_index()                      #Die 5 Betreiber mit den meisten Tankstellen auswählen und Index zurücksetzen
top5.columns = ['Operator', 'Number of Stations']       #Spalten verständlich umbenennen
top5.index = top5.index + 1                             #Ranking bei 1 statt bei 0 starten
print('Top 5 petrol station operators in Switzerland:') #Überschrift ausgeben
top5                                                    #Top-5-Rangliste anzeigen

## Visualise all petrol stations on an interactive map

In [ ]:
m = folium.Map(location=[47.41, 8.65], zoom_start=10, tiles='CartoDBPositron')  #Karte auf den Kanton Zürich zentrieren und Basiskarte festlegen

for _, row in df.iterrows():                                                    #Alle Tankstellen-Datensätze zeilenweise durchlaufen
    folium.Marker(                                                              #Marker für eine Tankstelle erstellen
        location=[row['lat'], row['lon']],                                      #Marker anhand von Breiten- und Längengrad platzieren
        icon=folium.Icon(color='green', icon='tint', prefix='fa'),              #Grünen Tankstellen-Marker mit Symbol festlegen
        popup=folium.Popup(                                                     #Popup mit Informationen zur Tankstelle erstellen
            f"<b>{row['name']}</b><br>Operator: {row['operator']}<br>Brand: {row['brand']}",  #Name, Betreiber und Marke im Popup anzeigen
            max_width=200                                                       #Maximale Breite des Popups festlegen
        )
    ).add_to(m)                                                                 #Marker zur Karte hinzufügen

m.save('petrol_stations_ch_map.html')                                           #Karte als HTML-Datei speichern

m                                                                               #Karte anzeigen

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')